# Import, config and load data

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import *

from darts import TimeSeries
from darts.dataprocessing.transformers import WindowTransformer, StaticCovariatesTransformer

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

import os
# -------------------- CONFIG --------------------
DATA_FOLDER       = "./data"
FIXED_DATA_PATH   = construct_path(DATA_FOLDER, "fixed")
DATASET_PATH      = construct_path(DATA_FOLDER, "dataset")

# Forecasting setup
OUTPUT_CHUNK_LEN  = 7      # how many days ahead each model predicts in one shot
INPUT_LAGS        = 7      # how many past days of the target the model sees
MULTI_MODELS      = True   # direct strategy: one sub-model per horizon step

# CV / split
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20   # enforced by split_series_list
CV_STRIDE                       = 1                   # For final evaluation No weekly stride, new forecast everyday


available_threads = get_available_threads()
print(f'CPU count: {available_threads}')

RANDOM_STATE      = 42
np.random.seed(RANDOM_STATE)

infra_types = ['health','education','residential','energy']
# intentions = ['intent','unintent']
intentions_oi = ['intent']
all_targets = [f"act_drone_infra_ua_{x}_{y}" for y in intentions_oi for x in infra_types]

CPU count: 16


In [2]:
regions,master_timeseries,regions_activity = \
    load_data(data_path=FIXED_DATA_PATH,dataset_path=DATASET_PATH)

master_timeseries shape: (847, 1622)
regions: 25  -  ['cherkasy', 'chernihiv', 'chernivtsi', 'dnipropetrovsk', 'donetsk'] ...
start: 2022-09-28 00:00:00 end: 2025-01-21 00:00:00


In [3]:
# print([x for x in master_timeseries.columns.tolist() if 'infra' in x])
# print(all_targets)

In [4]:
# A dataframe per infra type
damage_class_dfs, global_weather_columns = get_engineered_features_damageclassifiers(master_timeseries=master_timeseries,data_path=FIXED_DATA_PATH,targets_cols=all_targets,regions=regions,regions_activity=regions_activity)

damage_classes = dict()
for i,target in enumerate(all_targets):
    damage_classes[target] = {"raw_df" : damage_class_dfs[i]}


act_drone_infra_ua_health_intent
Amount of target =1: 48.000
Total rows: 16940.000
Target positive rate: 0.003
act_drone_infra_ua_education_intent
Amount of target =1: 69.000
Total rows: 16940.000
Target positive rate: 0.004
act_drone_infra_ua_residential_intent
Amount of target =1: 390.000
Total rows: 16940.000
Target positive rate: 0.023
act_drone_infra_ua_energy_intent
Amount of target =1: 123.000
Total rows: 16940.000
Target positive rate: 0.007


In [5]:
# For damage_classifiers
for key,value in damage_classes.items():
    holiday_cols_d, future_covariates_d, exclude_cols_d, past_covariates_d = split_future_and_past_cov(damage_classes[key]['raw_df'],global_weather_columns,f"{key}_binary")
    damage_classes[key]['split_future_and_past_cov'] = {
        'H': holiday_cols_d, 'F': future_covariates_d, 'E': exclude_cols_d, 'P': past_covariates_d
    }

# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85


### Building timeseries objects

In [6]:
# For damage classifiers
for key,value in damage_classes.items():
    target_series_list_d, past_covs_list_d,future_covs_list_d =build_ts_and_apply_window_transformer(
        for_global_reset=damage_classes[key]['raw_df'],
        target=f"{key}_binary",
        past_covariates=damage_classes[key]['split_future_and_past_cov']['P'],
        future_covariates=damage_classes[key]['split_future_and_past_cov']['F'],
        ed_alpha=halflife_to_alpha(7)
        )
    
    damage_classes[key]['build_ts_and_apply_window_transformer'] = {
        'T': target_series_list_d, 'P': past_covs_list_d, 'F': future_covs_list_d, 
    }

Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595


In [7]:
# For damage classifiers
for key,value in damage_classes.items():
    region_names, train_target_c, val_target_c, test_target_c, full_past_covs_c, full_fut_covs_c, target_for_cv_c, TRAIN_VAL_END,CV_START_VAL =\
      get_covs_and_encodings(
          damage_classes[key]['build_ts_and_apply_window_transformer']['T'],
          damage_classes[key]['build_ts_and_apply_window_transformer']['P'],
          damage_classes[key]['build_ts_and_apply_window_transformer']['F'],
          TRAIN_FRAC,VAL_FRAC
          )
    
    damage_classes[key]['get_covs_and_encodings'] = {
      'R': region_names, 
      'Tr': train_target_c,
      'V': val_target_c, 
      'Te':test_target_c, 
      'FPC':full_past_covs_c, 
      'FFC': full_fut_covs_c, 
      'TCV': target_for_cv_c, 
      'tve': TRAIN_VAL_END,
      'csv': CV_START_VAL
    }

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> ro

### Building the classifier and regressors themselves

In [8]:
from darts.models import LightGBMModel,LightGBMClassifierModel,SKLearnClassifierModel,CatBoostModel
from imbens.ensemble import SelfPacedEnsembleClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

COMMON_KWARGS = get_common_kwargs()

def get_damage_classifier(**kwargs):
    DTm_depth = kwargs.get('DTm_depth', 5)
    SPE_estm  = kwargs.get('SPE_estm', 100)
    clf = SelfPacedEnsembleClassifier(
        estimator    = DecisionTreeClassifier(max_depth=DTm_depth, random_state=RANDOM_STATE),
        n_estimators = SPE_estm,
        random_state = RANDOM_STATE,
        n_jobs       = available_threads,
    )
    return SKLearnClassifierModel(model=clf, **COMMON_KWARGS)

def get_feature_selector_classifier(**kwargs):
    return LightGBMClassifierModel(
        **COMMON_KWARGS,
        objective    = "binary",
        is_unbalance = True,
        random_state = RANDOM_STATE,
        **kwargs
    )


In [9]:
for key,value in damage_classes.items():
    damage_classifier_feature_selection = get_feature_selector_classifier()
    damage_classifier_feature_selection.fit(
        series          = damage_classes[key]['get_covs_and_encodings']['Tr'],
        past_covariates = damage_classes[key]['get_covs_and_encodings']['FPC'],
        future_covariates = damage_classes[key]['get_covs_and_encodings']['FFC'],
        #Train only on samples where the event of drone striking happened
    )
    top_100_features_d = get_top_100_from_lgbm(damage_classifier_feature_selection)
    top_100_features_dict_d = clean_feature_names(top_100_features_d)
    damage_classes[key]['feature_selection']  = {'top_100_features_d': top_100_features_d, 'top_100_features_dict_d': top_100_features_dict_d}

[LightGBM] [Info] Number of positive: 15, number of negative: 11445
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.321461 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 301045
[LightGBM] [Info] Number of data points in the train set: 11460, number of used features: 2093
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001309 -> initscore=-6.637258
[LightGBM] [Info] Start training from score -6.637258
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [10]:
for key,value in damage_classes.items():
    b1 = damage_classes[key]['feature_selection']['top_100_features_dict_d']
    b2 = damage_classes[key]['build_ts_and_apply_window_transformer']
    past_covs_list_d   = [subset_safe(ts, b1['pastcov_features_base']) for ts in b2['P']]
    future_covs_list_d = [subset_safe(ts, b1['futcov_features_base'])  for ts in b2['F']]
    _, train_target_d, val_target_d, test_target_d, full_past_covs_d, full_fut_covs_d, target_for_cv_d,TRAIN_VAL_END,CV_START_VAL = \
        get_covs_and_encodings(b2['T'],past_covs_list_d,future_covs_list_d,TRAIN_FRAC,VAL_FRAC)
    
    damage_classes[key]['get_covs_and_encodings'] = {
      'R': region_names, 
      'Tr': train_target_d,
      'V': val_target_d, 
      'Te':test_target_d, 
      'FPC':full_past_covs_d, 
      'FFC': full_fut_covs_d, 
      'TCV': target_for_cv_d, 
      'tve': TRAIN_VAL_END,
      'csv': CV_START_VAL
    }

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> ro

# Predictions
- Probability that a certain type of infrastructure gets damaged $\hat{\delta}(x) = P(X |D)$


## Cross-Validation

In [11]:
def _classif_metrics_hurdle(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).astype(float).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    def _safe(fn_):
        try: return float(fn_())
        except (ValueError, ZeroDivisionError): return np.nan
    return {
        "F1":        _safe(lambda: f1_score(y_true, y_pred, zero_division=0)),
        "Precision": _safe(lambda: precision_score(y_true, y_pred, zero_division=0)),
        "Recall":    _safe(lambda: recall_score(y_true, y_pred, zero_division=0)),
        "ROC_AUC":   _safe(lambda: roc_auc_score(y_true, y_prob)),
        "PR_AUC":    _safe(lambda: average_precision_score(y_true, y_prob)),
        "Brier":     _safe(lambda: brier_score_loss(y_true, y_prob)),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "n": int(len(y_true)),
    }


def evaluate_classif_long(long_df, threshold=0.5):
    def _rows(group_cols):
        rows = []
        for keys, sub in long_df.groupby(group_cols):
            m = _classif_metrics_hurdle(sub["y_true"], sub["y_prob"], threshold)
            key_map = (
                {group_cols: keys} if isinstance(group_cols, str)
                else dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
            )
            m.update(key_map)
            rows.append(m)
        return pd.DataFrame(rows)
    return {
        "per_region":         _rows("region").sort_values("F1", ascending=False).reset_index(drop=True),
        "per_horizon":        _rows("horizon").sort_values("horizon").reset_index(drop=True),
        "per_region_horizon": _rows(["region", "horizon"]).sort_values(["region", "horizon"]).reset_index(drop=True),
        "global":             _classif_metrics_hurdle(long_df["y_true"], long_df["y_prob"], threshold),
    }


In [12]:
def run_damage_cv(predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN):
    """Expanding-window CV for the damage classifiers.
    Directly predicts P(damage) for each infrastructure type.

    Returns
    -------
    fold_preds_d : dict[key -> list[list[TimeSeries]]]   P(damage) per type, per region
    """
    first_key = next(iter(damage_classes))
    ref_ts    = damage_classes[first_key]['get_covs_and_encodings']['TCV'][0]
    n_total   = len(ref_ts)
    start_idx = int(damage_classes[first_key]['get_covs_and_encodings']['csv'] * n_total)
    n_regions = len(damage_classes[first_key]['get_covs_and_encodings']['TCV'])

    fold_preds_d = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    n_preds    = 0
    n_retrains = 0
    damage_clfs = {key: None for key in damage_classes}

    for t0 in range(start_idx, n_total - OUTPUT_CHUNK_LEN + 1, predict_stride):
        steps_since_start = t0 - start_idx

        if steps_since_start % retrain_stride == 0:
            retrain_time = ref_ts.time_index[t0]
            for key in damage_classes:
                dc      = damage_classes[key]['get_covs_and_encodings']
                train_d = [ts.drop_after(retrain_time) for ts in dc['TCV']]
                dmg_clf = get_damage_classifier()
                dmg_clf.fit(
                    series            = train_d,
                    past_covariates   = dc['FPC'],
                    future_covariates = dc['FFC'],
                )
                damage_clfs[key] = dmg_clf
            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {retrain_time.date()})")

        split_time = ref_ts.time_index[t0]
        for key in damage_classes:
            dc            = damage_classes[key]['get_covs_and_encodings']
            pred_series_d = [ts.drop_after(split_time) for ts in dc['TCV']]
            preds_d = damage_clfs[key].predict(
                n                             = OUTPUT_CHUNK_LEN,
                series                        = pred_series_d,
                past_covariates               = dc['FPC'],
                future_covariates             = dc['FFC'],
                predict_likelihood_parameters = True,
                show_warnings                 = False,
            )
            for r_idx, pred_d in enumerate(preds_d):
                fold_preds_d[key][r_idx].append(pred_d.univariate_component(pred_d.n_components - 1))
        n_preds += 1

    print(f"   {n_preds} daily predictions, {n_retrains} retrains complete")
    return fold_preds_d

In [13]:
print("Running damage classifier CV ...")
fold_preds_d = run_damage_cv()

long_df_d = {}
for key in damage_classes:
    long_df_d[key] = (
        collect_predictions_long(damage_classes[key]['get_covs_and_encodings']['TCV'],
                                 fold_preds_d[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )
    print(f"  [{key}]  damage rows: {len(long_df_d[key]):,}")

This model will treat target `series` lagged values as numeric input features (and not categorical).


Running damage classifier CV ...


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 1  (data up to 2024-05-11)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


...

   retrain 10  (data up to 2024-07-13)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 11  (data up to 2024-07-20)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 12  (data up to 2024-07-27)
   79 daily predictions, 12 retrains complete
  [act_drone_infra_ua_health_intent]  damage rows: 11,060
  [act_drone_infra_ua_education_intent]  damage rows: 11,060
  [act_drone_infra_ua_residential_intent]  damage rows: 11,060
  [act_drone_infra_ua_energy_intent]  damage rows: 11,060


In [14]:
cv_classif_results_d = {}
for key in damage_classes:
    cv_classif_results_d[key] = evaluate_classif_long(long_df_d[key])
    gd    = cv_classif_results_d[key]["global"]
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Damage Classifier CV — Global ===")
    print(f"  F1      : {gd['F1']:.4f}  |  ROC-AUC : {gd['ROC_AUC']:.4f}  |  PR-AUC : {gd['PR_AUC']:.4f}  |  Brier : {gd['Brier']:.4f}")


=== [health] Damage Classifier CV — Global ===
  F1      : 0.0238  |  ROC-AUC : 0.8626  |  PR-AUC : 0.0168  |  Brier : 0.1856

=== [education] Damage Classifier CV — Global ===
  F1      : 0.0184  |  ROC-AUC : 0.7330  |  PR-AUC : 0.0120  |  Brier : 0.1947

=== [residential] Damage Classifier CV — Global ===
  F1      : 0.1466  |  ROC-AUC : 0.8351  |  PR-AUC : 0.1150  |  Brier : 0.1799

=== [energy] Damage Classifier CV — Global ===
  F1      : 0.0614  |  ROC-AUC : 0.6473  |  PR-AUC : 0.0319  |  Brier : 0.1980


In [15]:
for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Damage Classifier CV — Per-horizon ===")
    print(cv_classif_results_d[key]["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


=== [health] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.022676,0.904444,0.026744,0.195467,1580
1,2,0.026810,0.910349,0.214804,0.167969,1580
2,3,0.018141,0.889778,0.022569,0.192329,1580
3,4,0.020492,0.919175,0.024235,0.201468,1580
4,5,0.023981,0.800879,0.013532,0.183984,1580
5,6,0.026667,0.807126,0.013159,0.178499,1580
6,7,0.029197,0.848316,0.013502,0.179386,1580



=== [education] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.012903,0.704363,0.032602,0.178733,1580
1,2,0.017817,0.750635,0.016108,0.198923,1580
2,3,0.011364,0.709180,0.012403,0.183712,1580
3,4,0.029240,0.818273,0.020771,0.188794,1580
4,5,0.017738,0.731995,0.011736,0.211265,1580
5,6,0.010336,0.619235,0.013745,0.199130,1580
6,7,0.027714,0.787094,0.021517,0.202277,1580



=== [residential] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.148551,0.832096,0.105614,0.182880,1580
1,2,0.144177,0.852063,0.143407,0.183828,1580
2,3,0.134991,0.827235,0.120537,0.180794,1580
3,4,0.141791,0.816907,0.113646,0.183743,1580
4,5,0.148699,0.841207,0.142357,0.180143,1580
5,6,0.150659,0.841302,0.129746,0.178113,1580
6,7,0.158859,0.837948,0.110943,0.169460,1580



=== [energy] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.058824,0.700996,0.034126,0.200529,1580
1,2,0.046358,0.622228,0.027334,0.187588,1580
2,3,0.052941,0.607634,0.030216,0.193397,1580
3,4,0.059621,0.652822,0.057348,0.200238,1580
4,5,0.058981,0.640692,0.050108,0.201473,1580
5,6,0.093023,0.678791,0.044136,0.195564,1580
6,7,0.058824,0.637077,0.027144,0.207054,1580


## Classifier Calibration

In [16]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

_CAL_EPS = 1e-6

def _logit(p):
    p = np.clip(p, _CAL_EPS, 1 - _CAL_EPS)
    return np.log(p / (1 - p))

def fit_sigmoid_cal(y_prob, y_true):
    lr = LogisticRegression(C=1e6, solver="lbfgs")
    lr.fit(_logit(y_prob).reshape(-1, 1), np.asarray(y_true, dtype=int))
    return lr

def apply_sigmoid_cal(lr, y_prob):
    return lr.predict_proba(_logit(y_prob).reshape(-1, 1))[:, 1]

def _oof_one_group(y, p, method, n_splits, random_state):
    """OOF calibrated probs for one contiguous subset."""
    minority = int(min(np.bincount(y))) if set(y) == {0, 1} else 0
    n_splits_eff = max(2, min(n_splits, minority)) if minority >= 2 else 0
    if n_splits_eff < 2:
        return apply_sigmoid_cal(fit_sigmoid_cal(p, y), p)
    out = np.zeros_like(p)
    skf = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
    for fit_idx, eval_idx in skf.split(p, y):
            c = fit_sigmoid_cal(p[fit_idx], y[fit_idx])
            out[eval_idx] = apply_sigmoid_cal(c, p[eval_idx])
    return out


def oof_calibrated_probs(long_df, method, n_splits=5, random_state=RANDOM_STATE, group_col=None):
    """Returns (y_true, p_raw, p_calibrated) arrays aligned to long_df rows."""
    y = long_df["y_true"].to_numpy().astype(int)
    p = long_df["y_prob"].to_numpy().astype(float)
    if group_col is None:
        return y, p, _oof_one_group(y, p, method, n_splits, random_state)
    p_cal  = np.zeros_like(p)
    groups = long_df[group_col].to_numpy()
    for g in pd.unique(groups):
        mask = (groups == g)
        p_cal[mask] = _oof_one_group(y[mask], p[mask], method, n_splits, random_state)
    return y, p, p_cal


def fit_final_calibrators_per_horizon(long_df, method):
    """Fit one calibrator per horizon on all CV rows. Returns {horizon: calibrator}."""
    out = {}
    for h, sub in long_df.groupby("horizon"):
        y_h = sub["y_true"].to_numpy().astype(int)
        p_h = sub["y_prob"].to_numpy().astype(float)
        if len(np.unique(y_h)) < 2:
            out[int(h)] = None
            continue
        out[int(h)] = fit_sigmoid_cal(p_h, y_h)
    return out


def apply_calibrator_per_horizon(long_df, calibrators_per_h, method):
    """Apply per-horizon calibrator to y_prob column. Returns calibrated prob array."""
    p_raw = long_df["y_prob"].to_numpy().astype(float)
    out   = p_raw.copy()
    for h, sub in long_df.groupby("horizon"):
        cal = calibrators_per_h.get(int(h))
        if cal is None:
            continue
        idx   = sub.index.to_numpy()
        sub_p = sub["y_prob"].to_numpy().astype(float)
        out[idx] =apply_sigmoid_cal(cal, sub_p)
    return out


In [17]:
for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== Damage classifier [{label}] ===")
    df_reset = long_df_d[key].reset_index(drop=True)
    for method in ["sigmoid"]:
        y_true_cal, y_raw, y_cal = oof_calibrated_probs(df_reset, method=method, group_col="horizon")
        g_raw = _classif_metrics_hurdle(y_true_cal, y_raw)
        g_cal = _classif_metrics_hurdle(y_true_cal, y_cal)
        print(f"  {method:10s}  Brier {g_raw['Brier']:.4f} -> {g_cal['Brier']:.4f} "
              f"| ROC-AUC {g_raw['ROC_AUC']:.4f} -> {g_cal['ROC_AUC']:.4f} "
              f"| PR-AUC {g_raw['PR_AUC']:.4f} -> {g_cal['PR_AUC']:.4f}")


=== Damage classifier [health] ===
  sigmoid     Brier 0.1856 -> 0.0034 | ROC-AUC 0.8626 -> 0.8567 | PR-AUC 0.0168 -> 0.0225

=== Damage classifier [education] ===
  sigmoid     Brier 0.1947 -> 0.0043 | ROC-AUC 0.7330 -> 0.6996 | PR-AUC 0.0120 -> 0.0089

=== Damage classifier [residential] ===
  sigmoid     Brier 0.1799 -> 0.0276 | ROC-AUC 0.8351 -> 0.8316 | PR-AUC 0.1150 -> 0.1125

=== Damage classifier [energy] ===
  sigmoid     Brier 0.1980 -> 0.0150 | ROC-AUC 0.6473 -> 0.6436 | PR-AUC 0.0319 -> 0.0296


In [18]:
CAL_METHOD = "sigmoid"

damage_calibrators = {}
long_df_d_cal      = {}

for key in damage_classes:
    label      = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    df_d_reset = long_df_d[key].reset_index(drop=True)
    damage_calibrators[key] = fit_final_calibrators_per_horizon(df_d_reset, method=CAL_METHOD)
    print(f"[{label}] fitted {sum(c is not None for c in damage_calibrators[key].values())}/{len(damage_calibrators[key])} calibrators")

    long_df_d_cal[key]           = df_d_reset.copy()
    long_df_d_cal[key]["y_prob"] = apply_calibrator_per_horizon(df_d_reset, damage_calibrators[key], method=CAL_METHOD)

    gd_raw = cv_classif_results_d[key]["global"]
    gd_cal = evaluate_classif_long(long_df_d_cal[key])["global"]
    print(f"  Brier   : {gd_raw['Brier']:.4f}  ->  {gd_cal['Brier']:.4f}")
    print(f"  ROC-AUC : {gd_raw['ROC_AUC']:.4f}  ->  {gd_cal['ROC_AUC']:.4f}")
    print(f"  PR-AUC  : {gd_raw['PR_AUC']:.4f}  ->  {gd_cal['PR_AUC']:.4f}")

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Calibrated Damage Classifier — Per-horizon ===")
    print(evaluate_classif_long(long_df_d_cal[key])["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])

[health] fitted 7/7 calibrators
  Brier   : 0.1856  ->  0.0034
  ROC-AUC : 0.8626  ->  0.8744
  PR-AUC  : 0.0168  ->  0.0412
[education] fitted 7/7 calibrators
  Brier   : 0.1947  ->  0.0043
  ROC-AUC : 0.7330  ->  0.7442
  PR-AUC  : 0.0120  ->  0.0127
[residential] fitted 7/7 calibrators
  Brier   : 0.1799  ->  0.0275
  ROC-AUC : 0.8351  ->  0.8355
  PR-AUC  : 0.1150  ->  0.1160
[energy] fitted 7/7 calibrators
  Brier   : 0.1980  ->  0.0150
  ROC-AUC : 0.6473  ->  0.6520
  PR-AUC  : 0.0319  ->  0.0329

=== [health] Calibrated Damage Classifier — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.904444,0.026744,0.003144,1580
1,2,0.0,0.910349,0.214804,0.002941,1580
2,3,0.0,0.889778,0.022569,0.003143,1580
3,4,0.0,0.919175,0.024235,0.003136,1580
4,5,0.0,0.800879,0.013532,0.003773,1580
5,6,0.0,0.807126,0.013159,0.003780,1580
6,7,0.0,0.848316,0.013502,0.003780,1580



=== [education] Calibrated Damage Classifier — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.704363,0.032602,0.003765,1580
1,2,0.0,0.750635,0.016108,0.003771,1580
2,3,0.0,0.709180,0.012403,0.003776,1580
3,4,0.0,0.818273,0.020771,0.004396,1580
4,5,0.0,0.731995,0.011736,0.004405,1580
5,6,0.0,0.619235,0.013745,0.005032,1580
6,7,0.0,0.787094,0.021517,0.005021,1580



=== [residential] Calibrated Damage Classifier — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.832096,0.105614,0.028203,1580
1,2,0.0,0.852063,0.143407,0.026874,1580
2,3,0.0,0.827235,0.120537,0.026820,1580
3,4,0.0,0.816907,0.113646,0.027608,1580
4,5,0.0,0.841207,0.142357,0.027671,1580
5,6,0.0,0.841302,0.129746,0.027795,1580
6,7,0.0,0.837948,0.110943,0.027403,1580



=== [energy] Calibrated Damage Classifier — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.701010,0.034126,0.014893,1580
1,2,0.0,0.622228,0.027334,0.014923,1580
2,3,0.0,0.607634,0.030216,0.014915,1580
3,4,0.0,0.652822,0.057348,0.014779,1580
4,5,0.0,0.640692,0.050108,0.014853,1580
5,6,0.0,0.678791,0.044136,0.015433,1580
6,7,0.0,0.637077,0.027144,0.014919,1580


## Final Test Set Evaluation

In [19]:
def run_final_test(predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN):
    """Evaluate damage classifiers on the held-out test set.
    Directly predicts P(damage) for each infrastructure type.

    Returns
    -------
    fold_preds_d  : dict[key -> list[list[TimeSeries]]]  P(damage) per type, per region
    full_target_d : dict[key -> list[TimeSeries]]        full damage targets per type
    """
    full_target_d = {}
    for key in damage_classes:
        gc = damage_classes[key]['get_covs_and_encodings']
        full_target_d[key] = [tr.append(vl).append(te)
                               for tr, vl, te in zip(gc['Tr'], gc['V'], gc['Te'])]

    first_key      = next(iter(damage_classes))
    ref_ts         = full_target_d[first_key][0]
    n_total        = len(ref_ts)
    test_start_idx = int(damage_classes[first_key]['get_covs_and_encodings']['tve'] * n_total)
    n_regions      = len(full_target_d[first_key])

    fold_preds_d = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    n_preds    = 0
    n_retrains = 0
    damage_clfs = {key: None for key in damage_classes}

    for t0 in range(test_start_idx, n_total - OUTPUT_CHUNK_LEN + 1, predict_stride):
        steps_since_start = t0 - test_start_idx

        if steps_since_start % retrain_stride == 0:
            retrain_time = ref_ts.time_index[t0]
            for key in damage_classes:
                dc      = damage_classes[key]['get_covs_and_encodings']
                train_d = [ts.drop_after(retrain_time) for ts in full_target_d[key]]
                dmg_clf = get_damage_classifier()
                dmg_clf.fit(
                    series            = train_d,
                    past_covariates   = dc['FPC'],
                    future_covariates = dc['FFC'],
                )
                damage_clfs[key] = dmg_clf
            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {retrain_time.date()})")

        split_time = ref_ts.time_index[t0]
        for key in damage_classes:
            dc            = damage_classes[key]['get_covs_and_encodings']
            pred_series_d = [ts.drop_after(split_time) for ts in full_target_d[key]]
            preds_d = damage_clfs[key].predict(
                n                             = OUTPUT_CHUNK_LEN,
                series                        = pred_series_d,
                past_covariates               = dc['FPC'],
                future_covariates             = dc['FFC'],
                predict_likelihood_parameters = True,
                show_warnings                 = False,
            )
            for r_idx, pred_d in enumerate(preds_d):
                fold_preds_d[key][r_idx].append(pred_d.univariate_component(pred_d.n_components - 1))
        n_preds += 1

    print(f"   {n_preds} daily predictions, {n_retrains} retrains complete")
    return fold_preds_d, full_target_d

In [20]:
print("Running final test evaluation (retrain every 7 days) ...")
fold_preds_d_test, full_target_d = run_final_test()

This model will treat target `series` lagged values as numeric input features (and not categorical).


Running final test evaluation (retrain every 7 days) ...


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 1  (data up to 2024-08-05)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


...

   retrain 22  (data up to 2024-12-30)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 23  (data up to 2025-01-06)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 24  (data up to 2025-01-13)
   164 daily predictions, 24 retrains complete


In [21]:
test_long_d = {}
for key in damage_classes:
    test_long_d[key] = (
        collect_predictions_long(full_target_d[key], fold_preds_d_test[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )

test_results_d = {}
for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    test_results_d[key] = evaluate_classif_long(test_long_d[key])
    gd = test_results_d[key]["global"]
    print(f"\n=== Final Test — [{label}] Damage Classifier (uncalibrated) ===")
    print(f"  F1      : {gd['F1']:.4f}  |  ROC-AUC : {gd['ROC_AUC']:.4f}  |  PR-AUC : {gd['PR_AUC']:.4f}  |  Brier : {gd['Brier']:.4f}")

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== Final Test — [{label}] Per-horizon ===")
    print(test_results_d[key]["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


=== Final Test — [health] Damage Classifier (uncalibrated) ===
  F1      : 0.0592  |  ROC-AUC : 0.8885  |  PR-AUC : 0.0471  |  Brier : 0.1377

=== Final Test — [education] Damage Classifier (uncalibrated) ===
  F1      : 0.0459  |  ROC-AUC : 0.7556  |  PR-AUC : 0.0253  |  Brier : 0.1619

=== Final Test — [residential] Damage Classifier (uncalibrated) ===
  F1      : 0.2329  |  ROC-AUC : 0.8218  |  PR-AUC : 0.1977  |  Brier : 0.1794

=== Final Test — [energy] Damage Classifier (uncalibrated) ===
  F1      : 0.0623  |  ROC-AUC : 0.7023  |  PR-AUC : 0.0441  |  Brier : 0.2171

=== Final Test — [health] Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.059379,0.895128,0.096464,0.140129,3280
1,2,0.057534,0.888331,0.064259,0.137578,3280
2,3,0.058055,0.871446,0.046470,0.132402,3280
3,4,0.061625,0.883726,0.049222,0.136957,3280
4,5,0.062827,0.901801,0.051758,0.142376,3280
5,6,0.063624,0.898740,0.042613,0.135440,3280
6,7,0.051414,0.882003,0.049095,0.139288,3280



=== Final Test — [education] Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.045822,0.760190,0.033724,0.160684,3280
1,2,0.047182,0.778752,0.036989,0.163212,3280
2,3,0.040595,0.742376,0.022545,0.159643,3280
3,4,0.041725,0.754442,0.026836,0.159672,3280
4,5,0.049415,0.761919,0.024512,0.166955,3280
5,6,0.049793,0.745649,0.030728,0.161355,3280
6,7,0.046579,0.743326,0.025118,0.161949,3280



=== Final Test — [residential] Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.228615,0.816323,0.180912,0.183755,3280
1,2,0.226331,0.822448,0.199437,0.182975,3280
2,3,0.231970,0.820923,0.182080,0.182104,3280
3,4,0.234654,0.829387,0.219381,0.175464,3280
4,5,0.234124,0.823467,0.204464,0.176740,3280
5,6,0.233333,0.823821,0.239210,0.177684,3280
6,7,0.241590,0.816470,0.204243,0.177170,3280



=== Final Test — [energy] Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.052675,0.662844,0.039302,0.224468,3280
1,2,0.054432,0.674668,0.054361,0.226044,3280
2,3,0.061261,0.705429,0.049328,0.213016,3280
3,4,0.066968,0.708454,0.050736,0.213692,3280
4,5,0.064044,0.696531,0.048808,0.213103,3280
5,6,0.074364,0.732272,0.043893,0.209536,3280
6,7,0.065371,0.740298,0.050359,0.219595,3280


In [22]:
test_long_d_cal = {}
for key in damage_classes:
    label      = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    df_d_reset = test_long_d[key].reset_index(drop=True)
    test_long_d_cal[key]           = df_d_reset.copy()
    test_long_d_cal[key]["y_prob"] = apply_calibrator_per_horizon(
        df_d_reset, damage_calibrators[key], method=CAL_METHOD
    )
    gd_unc = test_results_d[key]["global"]
    gd_cal = evaluate_classif_long(test_long_d_cal[key])["global"]
    print(f"\n=== [{label}] Damage Classifier after {CAL_METHOD} calibration ===")
    print(f"  Brier   : {gd_unc['Brier']:.4f}  ->  {gd_cal['Brier']:.4f}")
    print(f"  ROC-AUC : {gd_unc['ROC_AUC']:.4f}  ->  {gd_cal['ROC_AUC']:.4f}")
    print(f"  PR-AUC  : {gd_unc['PR_AUC']:.4f}  ->  {gd_cal['PR_AUC']:.4f}")
    print(f"\n=== [{label}] Calibrated — Per-horizon ===")
    print(evaluate_classif_long(test_long_d_cal[key])["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


=== [health] Damage Classifier after sigmoid calibration ===
  Brier   : 0.1377  ->  0.0079
  ROC-AUC : 0.8885  ->  0.8792
  PR-AUC  : 0.0471  ->  0.0484

=== [health] Calibrated — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.895128,0.096464,0.008042,3280
1,2,0.0,0.888331,0.064259,0.008038,3280
2,3,0.0,0.871446,0.046470,0.008122,3280
3,4,0.0,0.883726,0.049222,0.008127,3280
4,5,0.0,0.901801,0.051758,0.007779,3280
5,6,0.0,0.898740,0.042613,0.007498,3280
6,7,0.0,0.882003,0.049095,0.007498,3280



=== [education] Damage Classifier after sigmoid calibration ===
  Brier   : 0.1619  ->  0.0087
  ROC-AUC : 0.7556  ->  0.7533
  PR-AUC  : 0.0253  ->  0.0245

=== [education] Calibrated — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.760190,0.033724,0.008714,3280
1,2,0.0,0.778752,0.036989,0.008724,3280
2,3,0.0,0.742376,0.022545,0.008753,3280
3,4,0.0,0.754442,0.026836,0.008739,3280
4,5,0.0,0.761919,0.024512,0.008746,3280
5,6,0.0,0.745649,0.030728,0.008744,3280
6,7,0.0,0.743326,0.025118,0.008766,3280



=== [residential] Damage Classifier after sigmoid calibration ===
  Brier   : 0.1794  ->  0.0496
  ROC-AUC : 0.8218  ->  0.8217
  PR-AUC  : 0.1977  ->  0.1982

=== [residential] Calibrated — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.816323,0.180912,0.048882,3280
1,2,0.0,0.822448,0.199437,0.049337,3280
2,3,0.0,0.820923,0.182080,0.049758,3280
3,4,0.0,0.829387,0.219381,0.049400,3280
4,5,0.0,0.823467,0.204464,0.049453,3280
5,6,0.0,0.823821,0.239210,0.049581,3280
6,7,0.0,0.816470,0.204243,0.050746,3280



=== [energy] Damage Classifier after sigmoid calibration ===
  Brier   : 0.2171  ->  0.0170
  ROC-AUC : 0.7023  ->  0.7017
  PR-AUC  : 0.0441  ->  0.0429

=== [energy] Calibrated — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.662844,0.039302,0.016386,3280
1,2,0.0,0.674668,0.054361,0.016907,3280
2,3,0.0,0.705429,0.049328,0.016896,3280
3,4,0.0,0.708454,0.050736,0.017146,3280
4,5,0.0,0.696531,0.048808,0.017467,3280
5,6,0.0,0.732272,0.043893,0.017198,3280
6,7,0.0,0.740298,0.050359,0.017158,3280
